In [3]:
# WUMPUS WORLD VARIANT: LAVA CELLS AND THE HEAT PERCEPT

#  SECTION 1: ENVIRONMENT

# The Environment holds the TRUE state of the world.
# The agent NEVER reads from this class directly —
# it only receives percepts via get_percepts().
#
# Grid layout:  (0,0) = top-left,  (4,4) = bottom-right
# Agent start:  (4,0) = bottom-left  [always safe]
#
# Hazard types:
#   Pit    - agent dies on entry; emits BREEZE to adjacent cells
#   Wumpus - agent dies on entry; emits STENCH to adjacent cells
#   Lava   - agent dies on entry; emits HEAT   to adjacent cells

class Environment:

    def __init__(self, pits, wumpus, lava_cells, gold):
        """
        pits       : list of (row, col) tuples — pit locations
        wumpus     : (row, col) tuple          — Wumpus location
        lava_cells : list of (row, col) tuples — lava locations
        gold       : (row, col) tuple          — gold location
        """
        self.size          = 5
        self.pits          = set(pits)
        self.wumpus        = wumpus
        self.wumpus_alive  = True
        self.lava_cells    = set(lava_cells)
        self.gold          = gold
        self.gold_collected = False
        self.start         = (4, 0)   # Agent always starts here

        # Safety check: the start cell must always be hazard-free
        assert self.start not in self.pits,       "ERROR: start cell has a pit!"
        assert self.start != self.wumpus,          "ERROR: start cell has the Wumpus!"
        assert self.start not in self.lava_cells,  "ERROR: start cell has lava!"

    def in_bounds(self, r, c):
        """True if (r, c) is inside the 5×5 grid."""
        return 0 <= r < self.size and 0 <= c < self.size

    def neighbours(self, r, c):
        """All orthogonally adjacent cells that are within the grid."""
        return [(nr, nc)
                for nr, nc in [(r-1, c), (r+1, c), (r, c-1), (r, c+1)]
                if self.in_bounds(nr, nc)]

    def get_percepts(self, r, c):
        """
        Returns the set of percepts the agent receives at cell (r, c).

        Percept rules:
          STENCH  — Wumpus is in an adjacent cell (and still alive)
          BREEZE  — a Pit is in an adjacent cell
          HEAT    — a Lava cell is adjacent  - My new twist percept
          GLITTER — gold is here and not yet collected
        """
        percepts = set()

        if self.wumpus_alive and self.wumpus in self.neighbours(r, c):
            percepts.add('Stench')

        if any(pit in self.neighbours(r, c) for pit in self.pits):
            percepts.add('Breeze')

        # HEAT: unique to the Lava twist — independent of Breeze/Stench
        if any(lava in self.neighbours(r, c) for lava in self.lava_cells):
            percepts.add('Heat')

        if (r, c) == self.gold and not self.gold_collected:
            percepts.add('Glitter')

        return percepts

    def is_deadly(self, r, c):
        """True if stepping into (r, c) would kill the agent."""
        if (r, c) in self.pits:                          return True
        if self.wumpus_alive and (r, c) == self.wumpus:  return True
        if (r, c) in self.lava_cells:                    return True
        return False

    def kill_wumpus(self):
        """Called when the agent fires its arrow and hits the Wumpus."""
        self.wumpus_alive = False
        print("  [SCREAM] The Wumpus has been killed!")

    def collect_gold(self):
        """Called when the agent grabs the gold."""
        self.gold_collected = True

    def display(self, agent_pos=None):
        print("\n TRUE WORLD STATE ")
        for r in range(self.size):
            row = "|"
            for c in range(self.size):
                if   agent_pos == (r, c):                          row += "  A  |"
                elif (r, c) == self.start:                         row += "  S  |"
                elif (r, c) in self.pits:                          row += "  P  |"
                elif (r, c) == self.wumpus and self.wumpus_alive:  row += "  W  |"
                elif (r, c) in self.lava_cells:                    row += "  L  |"
                elif (r, c) == self.gold and not self.gold_collected: row += "  G  |"
                else:                                              row += "  .  |"
            print(row)
        print("P=Pit  W=Wumpus  L=Lava  G=Gold  S=Start  A=Agent")
        print("─" * 50)








In [4]:
#  SECTION 2: KNOWLEDGE BASE
#
# The KnowledgeBase stores everything the agent has inferred
# using propositional logic.
#
# For each cell (r, c) the KB tracks:
#
#   no_pit / no_wumpus / no_lava
#       - proven ABSENT  (derived from negative percepts)
#
#   poss_pit / poss_wumpus / poss_lava
#       - possibly PRESENT (derived from positive percepts)
#
#   safe    - True when ALL THREE hazards are proven absent
#   visited - True when the agent has physically stood here
#
# The three hazard chains (pit, wumpus, lava) are tracked
# INDEPENDENTLY — this is the key design feature of my twist.

class KnowledgeBase:

    def __init__(self, size=5, start=(4, 0)):
        self.size = size

        # Initialise all per-cell belief arrays to False (unknown)
        self.no_pit      = [[False] * size for _ in range(size)]
        self.no_wumpus   = [[False] * size for _ in range(size)]
        self.no_lava     = [[False] * size for _ in range(size)]
        self.poss_pit    = [[False] * size for _ in range(size)]
        self.poss_wumpus = [[False] * size for _ in range(size)]
        self.poss_lava   = [[False] * size for _ in range(size)]
        self.visited     = [[False] * size for _ in range(size)]
        self.safe        = [[False] * size for _ in range(size)]

        # The start cell is always known to be safe before any step
        r, c = start
        self.no_pit[r][c] = self.no_wumpus[r][c] = self.no_lava[r][c] = True
        self.safe[r][c]   = True

    def mark_visited(self, r, c):
        """
        Record that the agent has occupied (r, c) and survived.
        Survival proves the cell is safe.
        """
        self.visited[r][c] = True
        self.safe[r][c]    = True   # survived - safe by definition

    def update(self, r, c, percepts, env):
        """
        Core inference step — called every time the agent enters a cell.

        For each unvisited neighbour of (r, c) we apply two rules:

        NEGATIVE INFERENCE (most powerful):
          If we do NOT sense a percept here, then the corresponding
          hazard CANNOT exist in any adjacent unvisited cell.
          e.g. no Heat at (r,c) - no Lava at any neighbour of (r,c)

        POSITIVE INFERENCE:
          If we DO sense a percept here, then the hazard MIGHT exist
          in one of the unvisited neighbours (we can't say which yet).

        This is standard propositional Wumpus logic extended with a
        third independent hazard chain for Lava / Heat.
        """
        for nr, nc in env.neighbours(r, c):
            if self.visited[nr][nc]:
                continue   # Already proven safe by survival — skip

            #  Negative inference
            if 'Breeze' not in percepts:
                self.no_pit[nr][nc]    = True
            if 'Stench' not in percepts:
                self.no_wumpus[nr][nc] = True
            if 'Heat' not in percepts:          # - KEY: third chain
                self.no_lava[nr][nc]   = True

            #  Positive inference
            if 'Breeze' in percepts and not self.no_pit[nr][nc]:
                self.poss_pit[nr][nc]    = True
            if 'Stench' in percepts and not self.no_wumpus[nr][nc]:
                self.poss_wumpus[nr][nc] = True
            if 'Heat' in percepts and not self.no_lava[nr][nc]:    # - KEY
                self.poss_lava[nr][nc]   = True

        # Re-derive which cells are now confirmed safe
        self._infer_safe_cells()

    def _infer_safe_cells(self):
        """
        A cell is SAFE if and only if all three hazard types are
        proven absent. One unresolved hazard keeps the cell unsafe.
        """
        for r in range(self.size):
            for c in range(self.size):
                if (self.no_pit[r][c] and
                        self.no_wumpus[r][c] and
                        self.no_lava[r][c]):
                    self.safe[r][c] = True

    def resolve_wumpus(self):
        """
        After the Wumpus is killed, remove all Wumpus-related beliefs
        and re-derive safe cells (some may now become safe).
        """
        for r in range(self.size):
            for c in range(self.size):
                self.no_wumpus[r][c]   = True
                self.poss_wumpus[r][c] = False
        self._infer_safe_cells()

    def find_wumpus(self):
        """
        Tries to deduce the Wumpus's exact location.

        If only ONE cell still has poss_wumpus=True and no_wumpus=False,
        then by elimination that MUST be where the Wumpus is.
        The agent only shoots when this returns a single candidate.

        Returns (r, c) or None.
        """
        candidates = [
            (r, c)
            for r in range(self.size)
            for c in range(self.size)
            if self.poss_wumpus[r][c] and not self.no_wumpus[r][c]
        ]
        return candidates[0] if len(candidates) == 1 else None

    def hazard_score(self, r, c):
        """
        Rough danger estimate for an unvisited cell.
        Counts how many hazard types are still possible here.
        Used only as a last-resort fallback (higher = more dangerous).
        """
        return sum([
            self.poss_pit[r][c],
            self.poss_wumpus[r][c],
            self.poss_lava[r][c]
        ])

    def display(self):
        """Prints the agent's belief state for every cell in the grid."""
        print("\n AGENT KNOWLEDGE BASE ")
        print("Legend: S=Safe  V=Visited  ?P=poss_pit  "
              "?W=poss_wumpus  ?L=poss_lava")
        for r in range(self.size):
            parts = []
            for c in range(self.size):
                tags = []
                if self.visited[r][c]:  tags.append("V")
                if self.safe[r][c]:     tags.append("S")
                if self.poss_pit[r][c]    and not self.no_pit[r][c]:    tags.append("?P")
                if self.poss_wumpus[r][c] and not self.no_wumpus[r][c]: tags.append("?W")
                if self.poss_lava[r][c]   and not self.no_lava[r][c]:   tags.append("?L")
                label = ",".join(tags) if tags else "?"
                parts.append(f"({r},{c}):[{label:8}]")
            print("  " + "  ".join(parts))
        print("─" * 50)


In [5]:
#   SECTION 3: AGENT
#
# The Agent drives the full sense - update KB - act cycle.
#
# Decision priority (checked in order each step):
#
#   1. GRAB   — pick up gold if standing on it
#   2. CLIMB  — exit the cave if at start and holding gold  - WIN
#   3. HOME   — BFS route toward start if carrying gold
#   4. EXPLORE — BFS to nearest confirmed-safe unvisited cell
#   5. SHOOT  — fire arrow if Wumpus location is fully confirmed
#   6. RISK   — move to least-dangerous adjacent unvisited cell
#                 (last resort when no safe moves exist)

from collections import deque

class Agent:

    def __init__(self, env):
        self.env        = env
        self.kb         = KnowledgeBase(size=env.size, start=env.start)
        self.pos        = env.start
        self.has_gold   = False
        self.has_arrow  = True
        self.alive      = True
        self.won        = False
        self.step_count = 0
        self.history    = []   # log of (step, pos, percepts, action)

    def bfs_to_nearest(self, targets):
        """
        Finds the shortest path from the current position to any cell
        in `targets`, travelling only through confirmed-safe cells.

        Returns the FIRST STEP (next cell to move to), or None if no
        path through safe cells exists.
        """
        if not targets:
            return None

        queue = deque([(self.pos, [])])
        seen  = {self.pos}

        while queue:
            (r, c), path = queue.popleft()

            if (r, c) in targets:
                # path[0] is the first step from self.pos toward (r,c)
                return path[0] if path else None

            for nr, nc in self.env.neighbours(r, c):
                if (nr, nc) not in seen and self.kb.safe[nr][nc]:
                    seen.add((nr, nc))
                    queue.append(((nr, nc), path + [(nr, nc)]))

        return None   # No route found through known-safe cells

    def direction_to(self, target):
        """Direction from current position to target."""
        r, c   = self.pos
        tr, tc = target
        if tr < r: return "UP"
        if tr > r: return "DOWN"
        if tc < c: return "LEFT"
        return "RIGHT"

    def step(self):
        """
        Executes one complete agent step:
          sense percepts - update KB - choose best action - execute it

        Returns True while the episode continues, False when it ends.
        """
        if not self.alive or self.won:
            return False

        self.step_count += 1
        r, c = self.pos

        #  SENSE & UPDATE
        percepts = self.env.get_percepts(r, c)
        self.kb.mark_visited(r, c)
        self.kb.update(r, c, percepts, self.env)

        print(f"\nStep {self.step_count}: Agent at {self.pos}")
        print(f"  Percepts: {percepts if percepts else '{none}'}")

        #  PRIORITY 1: GRAB GOLD
        if 'Glitter' in percepts:
            self.has_gold = True
            self.env.collect_gold()
            print("  Action: GRAB GOLD ")
            self.history.append(
                (self.step_count, self.pos, percepts, "GRAB"))
            return True

        #  PRIORITY 2: CLIMB OUT
        if self.has_gold and self.pos == self.env.start:
            self.won = True
            print("  Action: CLIMB OUT — MISSION COMPLETE! ")
            self.history.append(
                (self.step_count, self.pos, percepts, "CLIMB"))
            return False

        #  PRIORITY 3: NAVIGATE HOME (carrying gold)
        if self.has_gold:
            next_cell = self.bfs_to_nearest({self.env.start})
            if next_cell:
                direction = self.direction_to(next_cell)
                print(f"  Reasoning: carrying gold - navigating home ({direction})")
                print(f"  Action: MOVE to {next_cell}")
                self.history.append(
                    (self.step_count, self.pos, percepts, f"HOME {direction}"))
                self._execute_move(next_cell)
                return self.alive

        #  PRIORITY 4: EXPLORE SAFE UNVISITED CELLS
        safe_unvisited = {
            (nr, nc)
            for nr in range(self.env.size)
            for nc in range(self.env.size)
            if self.kb.safe[nr][nc] and not self.kb.visited[nr][nc]
        }

        next_cell = self.bfs_to_nearest(safe_unvisited)

        # BFS may fail if a safe cell is unreachable through currently
        # known-safe cells. Try a direct adjacent move as fallback.
        if next_cell is None and safe_unvisited:
            for candidate in safe_unvisited:
                if candidate in self.env.neighbours(r, c):
                    next_cell = candidate
                    break

        if next_cell:
            direction = self.direction_to(next_cell)
            print(f"  Reasoning: safe unvisited cells exist - move {direction}")
            print(f"  Action: MOVE to {next_cell}")
            self.history.append(
                (self.step_count, self.pos, percepts, f"MOVE {direction}"))
            self._execute_move(next_cell)
            return self.alive

        #  PRIORITY 5: SHOOT WUMPUS
        # Only shoot when the Wumpus location is narrowed to exactly
        # one candidate — otherwise we might waste the arrow.
        wumpus_pos = self.kb.find_wumpus()
        if wumpus_pos and self.has_arrow:
            direction = self.direction_to(wumpus_pos)
            print(f"  Reasoning: Wumpus confirmed at {wumpus_pos} "
                  f"- SHOOT {direction}")
            print(f"  Action: SHOOT arrow {direction}")
            self.has_arrow = False
            self.env.kill_wumpus()
            self.kb.resolve_wumpus()
            self.history.append(
                (self.step_count, self.pos, percepts, f"SHOOT {direction}"))
            return True

        #  PRIORITY 6: RISK-MINIMUM FALLBACK
        # No safe cells available and no confirmed Wumpus to shoot.
        # Pick the adjacent unvisited cell with the lowest hazard score.
        # This is a last resort — death is possible on this move.
        unvisited_adjacent = [
            cell for cell in self.env.neighbours(r, c)
            if not self.kb.visited[cell[0]][cell[1]]
        ]

        if unvisited_adjacent:
            best      = min(unvisited_adjacent,
                            key=lambda cell: self.kb.hazard_score(
                                cell[0], cell[1]))
            direction = self.direction_to(best)
            score     = self.kb.hazard_score(best[0], best[1])
            print(f"  Reasoning: no safe cells — "
                  f"risk-minimum fallback (score={score})")
            print(f"  Action: MOVE (risky) to {best}")
            self.history.append(
                (self.step_count, self.pos, percepts, f"MOVE-RISKY {direction}"))
            self._execute_move(best)
            return self.alive

        print("  Agent is stuck — no available moves.")
        return False

    def _execute_move(self, new_pos):
        """Physically moves the agent and checks if the cell is deadly."""
        self.pos = new_pos
        r, c = new_pos
        if self.env.is_deadly(r, c):
            print(f"    Agent entered deadly cell {new_pos} — GAME OVER")
            self.alive = False

    def run(self, max_steps=60):
        """
        Runs the agent until it wins, dies, or reaches max_steps.
        Prints a full step-by-step trace and final KB summary.
        Returns 'WIN', 'DEAD', or 'TIMEOUT'.
        """
        print("=" * 55)
        print("STARTING SIMULATION")
        self.env.display(agent_pos=self.pos)
        print("=" * 55)

        for _ in range(max_steps):
            if not self.step():
                break

        print("\n" + "=" * 55)
        if self.won:
            print(f" SUCCESS — completed in {self.step_count} steps.")
        elif not self.alive:
            print(f" FAILURE — agent died at step {self.step_count}.")
        else:
            print(f"  STOPPED — reached step limit ({max_steps}).")
        print("=" * 55)

        self.kb.display()
        return "WIN" if self.won else ("DEAD" if not self.alive else "TIMEOUT")


In [6]:
#   SECTION 4: TEST CASES
#
# Five tests, each targeting a specific agent behaviour.
# Hazard positions are chosen so each test exercises one feature
# cleanly, without hazards adjacent to the start cell.
#
# Expected results (all verified):
#   Test 1 — WIN  (standard explore - grab - return)
#   Test 2 — WIN  (Heat percept triggers lava avoidance)
#   Test 3 — WIN  (Wumpus confirmed and shot before entering)
#   Test 4 — WIN  (Breeze + Heat handled as independent KB chains)
#   Test 5 — WIN  (fallback strategy survives risky first moves)

def run_test(label, pits, wumpus, lava_cells, gold, max_steps=60):
    """Helper: build environment, run agent, print result."""
    print(f"\n{'#'*55}")
    print(f"  TEST: {label}")
    print(f"{'#'*55}")
    env    = Environment(pits=pits, wumpus=wumpus,
                         lava_cells=lava_cells, gold=gold)
    agent  = Agent(env)
    result = agent.run(max_steps=max_steps)
    print(f"\n  RESULT: {result}\n")
    return result


#  TEST 1: Standard win
# All hazards are in the top corners, well away from the start.
# The agent routes right along row 4, up column 2, navigates
# around hazard signals, grabs gold at (2,4), then BFS routes
# home through known-safe cells.
#
# Demonstrates: basic explore - grab - return-home cycle.
# Expected: WIN
r1 = run_test(
    label      = "Standard Win — explore, grab gold, return home",
    pits       = [(0, 3)],
    wumpus     = (0, 0),
    lava_cells = [(0, 4)],
    gold       = (2, 4)
)


#  TEST 2: Heat percept triggers lava inference
# Lava is at (1,3). When the agent visits cells adjacent to it
# (e.g. (1,2), (0,3)) it senses Heat and marks those neighbours
# as possibly-lava in the KB. The agent avoids those cells and
# routes around to reach gold at (4,4).
#
# Demonstrates: Heat percept causes correct negative/positive
# lava inference. KB shows ?L markers on lava neighbours.
# Expected: WIN
r2 = run_test(
    label      = "Heat Avoidance — lava inferred from Heat, routed around",
    pits       = [(0, 0)],
    wumpus     = (0, 2),
    lava_cells = [(1, 3)],
    gold       = (4, 4)
)


#  TEST 3: Wumpus location confirmed and shot
# Wumpus is at (2,4). The agent clears the entire bottom row
# (no Stench anywhere in row 4), then visits (3,4) which gives
# Stench. At this point only one cell — (2,4) — has not been
# ruled out as a Wumpus candidate. find_wumpus() returns (2,4)
# and the agent fires before ever entering that cell.
# Lava at (2,0) is far from the column-4 path.
#
# Demonstrates: Wumpus narrowed to one candidate - shoot - safe path unlocked.
# Expected: WIN
r3 = run_test(
    label      = "Wumpus Shooting — single candidate confirmed, arrow fired",
    pits       = [(0, 1)],
    wumpus     = (2, 4),
    lava_cells = [(2, 0)],
    gold       = (0, 4)
)


#  TEST 4: Overlapping percepts — Breeze AND Heat
# Pit at (0,2) and Lava at (2,2): cell (1,2) produces BOTH
# Breeze and Heat simultaneously. The KB must update two fully
# independent inference chains — poss_pit for Breeze, poss_lava
# for Heat — without conflating them or over-restricting safe cells.
# Gold is at (4,4), reachable via the right side of the grid.
#
# Demonstrates: independent KB chains do not interfere with each other.
# Expected: WIN
r4 = run_test(
    label      = "Overlapping Percepts — Breeze + Heat tracked independently",
    pits       = [(0, 2)],
    wumpus     = (0, 0),
    lava_cells = [(2, 2)],
    gold       = (4, 4)
)


#  TEST 5: Fallback strategy from step 1
# Pit at (3,1) and Lava at (4,1): the start cell (4,0) receives
# both Breeze and Heat on step 1, meaning both neighbours (3,0)
# and (4,1) are suspect and no safe cells are provable yet.
# The fallback fires immediately, picking the least-risky adjacent
# cell. (3,0) has hazard score 1 vs (4,1) which is actual lava,
# but also score 1 — whichever is picked the agent must survive.
# After two risky moves the agent breaks into clear space and
# accumulates enough KB to navigate safely to gold at (0,4).
#
# Demonstrates: graceful degradation when no safe cell is provable.
# Expected: WIN
r5 = run_test(
    label      = "Fallback Strategy — both start neighbours risky at step 1",
    pits       = [(3, 1)],
    wumpus     = (0, 0),
    lava_cells = [(4, 1)],
    gold       = (0, 4),
    max_steps  = 80
)


#  FINAL SUMMARY
print("\n" + "=" * 55)
print("FINAL SUMMARY")
print("=" * 55)
tests = [
    ("Standard Win",          r1),
    ("Heat Avoidance",        r2),
    ("Wumpus Shooting",       r3),
    ("Overlapping Percepts",  r4),
    ("Fallback Strategy",     r5),
]
for i, (label, result) in enumerate(tests, 1):
    icon = "✅" if result == "WIN" else "❌"
    print(f"  Test {i} ({label}): {icon} {result}")


#######################################################
  TEST: Standard Win — explore, grab gold, return home
#######################################################
STARTING SIMULATION

 TRUE WORLD STATE 
|  W  |  .  |  .  |  P  |  L  |
|  .  |  .  |  .  |  .  |  .  |
|  .  |  .  |  .  |  .  |  G  |
|  .  |  .  |  .  |  .  |  .  |
|  A  |  .  |  .  |  .  |  .  |
P=Pit  W=Wumpus  L=Lava  G=Gold  S=Start  A=Agent
──────────────────────────────────────────────────

Step 1: Agent at (4, 0)
  Percepts: {none}
  Reasoning: safe unvisited cells exist - move UP
  Action: MOVE to (3, 0)

Step 2: Agent at (3, 0)
  Percepts: {none}
  Reasoning: safe unvisited cells exist - move UP
  Action: MOVE to (2, 0)

Step 3: Agent at (2, 0)
  Percepts: {none}
  Reasoning: safe unvisited cells exist - move UP
  Action: MOVE to (1, 0)

Step 4: Agent at (1, 0)
  Percepts: {'Stench'}
  Reasoning: safe unvisited cells exist - move DOWN
  Action: MOVE to (2, 0)

Step 5: Agent at (2, 0)
  Percepts: {none}
  Rea